In [1]:
import pandas as pd
import numpy as np
import re
import os

In [2]:
df = pd.read_parquet('./news_data.parquet', engine='pyarrow')

In [3]:
# Checking those full_text with more than 20000 tokens
df[df['full_text'].apply(lambda x: len(x.split()) > 20000)].index

Index([3814, 9071, 16692], dtype='int64')

In [51]:
import re
import pandas as pd
from nltk.tokenize import sent_tokenize
import nltk

nltk.download('punkt', force=True)


def custom_conservative_cleaner(text: str) -> str:
    """
    Clean article or social media text by removing noise, UI instructions, trackers,
    cookie banners, timestamp artifacts, templated sentences, and lengthy consent notices.

    The cleaning steps include:
      - Replacing non-breaking spaces and adding spaces in long-digit sequences.
      - Removing jumbled timestamps and truncating text after markers.
      - Removing block patterns, advertisement artifacts, and UI control symbols.
      - Filtering out sentences that match defined partial or exact patterns.
      - Removing consent/cookie disclosure sentences via specialized patterns.
      - All matching is done in a case-insensitive manner.

    Args:
        text (str): Raw input text.

    Returns:
        str: Cleaned version of the input text.
    """
    if not isinstance(text, str):
        return ""

    # Replace non-breaking spaces and ensure separation in long digit sequences
    text = text.replace("\xa0", " ")
    text = re.sub(r"(\d)(?=\d{3,})", r"\1 ", text)  # e.g., 1475 -> handled with space if adjacent digits

    # Remove jumbled timestamps like "Live00:0001:1601:16"
    text = re.sub(r"\b(?:Live)?(?:\d{1,2}:){1,4}\d{1,2}\b", " ", text)

    # Truncate from "Source: CNA/" onwards
    text = re.split(r"(?i)Source: CNA/", text)[0]

    # Remove block patterns (e.g., controls, ads, hyperlinks)
    patterns = [
        r"Press shift question mark to access a list of keyboard shortcuts.*?(?=Next Up|facebook|Over 25 buildings|$)",
        r"Keep Watching.*?(?=Next video in \d+ seconds|facebook|More Videos|$)",
        r"Next video in \d+ seconds.*?(?=facebook|More Videos|$)",
        r"0 seconds of \d+ minute[s]?, \d+ secondsVolume.*?(?=facebook|More Videos|$)",
        r"facebookxlinkedinLinkhttps?://[^\s]+Copied",
        r"More Videos.*?(?=A prominent|From \d{4}|This story has been shared|$)",
        r"This story has been shared \d+ times\.\d+",
        r"This story has \d+[Kk] comments\.\d+",
        r"Filed under.*",
        r"Read Next.*",
        r"SPONSORED STORIES.*?Around The Web",
        r"Around The Web.*?Powered by ZergNet",
        r"Now onPage Six.*?See AllVideo",
        r"VideoWill .*?\| The Injury Report",
        r"https?:\/\/[^\s]+",
        r"\d{6,}",
    ]
    for pattern in patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.DOTALL)

    # Remove inline UI control symbols and hotkeys
    tag_noise = [
        r"Open/Close/ or \?", r"Play/PauseSPACE", r"Increase Volume↑", r"Decrease Volume↓",
        r"Seek Forward→", r"Seek Backward←", r"Captions On/Offc", r"Fullscreen/Exit Fullscreenf",
        r"Mute/Unmutem", r"Decrease Caption Size-Increase Caption Size\+ or =", r"Seek %0-9"
    ]
    for tag in tag_noise:
        text = re.sub(tag, "", text)

    # Advertisement cleanup: break up merged advertisement strings
    text = re.sub(r"Advertisement(?=[A-Z])", "", text)
    text = re.sub(r"([a-z])Advertisement([A-Z])", r"\1 \2", text)
    text = re.sub(r"\bAdvertisement\b", "", text)

    # Phrase-level removals
    text = re.sub(r"(?i)getty images?", "", text)
    text = re.sub(r"\(File photo:[^)]+\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(?i)via\s*reuters", "", text)
    text = re.sub(r"(?i)(click here to|tap here to)[^.?!]*[.?!]", "", text)
    text = re.sub(r"\(Updated:\s*\d{1,2} [A-Za-z]{3,9} 20\d{2} \d{1,2}:\d{2}[APMapm]{2}\)", "", text)
    text = re.sub(r"(?i)bookmark\s*share\s*whats\s*app\s*telegram\s*facebook\s*twitter\s*email\s*linked\s*in", "", text)
    text = re.sub(r"(?i)facebook\s*twitter\s*whats\s*app\s*sms\s*email\s*print\s*copy\s*article\s*link", "", text)

    # Split sentences including newlines and punctuation endings
    sentence_candidates = re.split(r'(?<=[.!?])(?=\s|\Z)|[\n\r]+', text)

    # Partial keyword patterns for sentence removal (case-insensitive)
    partial_match_keywords = [
        r"Data Retention Period:",
        r"Purposes \(Consent\)",
        r"Purposes \(Legitimate Interest\)",
        r"Legitimate Interest Claim:http",
        r"Privacy Policy:http",
        r"Customise your consent preferences for Cookie Categories",
        r"Number of Vendors seeking consent: \d{1,4}",
        r"Number of Vendors seeking consent or relying on legitimate interest: \d{1,4}",
        r"(Privacy Policy:\s*){2,}",  # matches repeated occurrences like "Privacy Policy:" repeated at least twice
        r"Privacy Policy:",
        r"Features Privacy Policy:",
        r"Vendors Privacy Policy:",
        r"Features Special Features Privacy Policy:",
        r"Special Features Legitimate Interest Claim:",
        r"Legitimate Interest Claim:",
    ]

    # Exact sentence matches (normalized to lowercase for consistency)
    exact_matches = [
        "Special Purposes",
        "Data Categories",
        "Device Storage Overview",
        "Privacy Policy",
        "The cookies that are categorized as \"Necessary\" are stored on your browser as they are essential for enabling the basic functionalities of the site.",
        "These cookies do not store any personally identifiable data.",
        "Functional cookies help perform certain functionalities like sharing the content of the website on social media platforms, collecting feedback, and other third-party features.",
        "Analytical cookies are used to understand how visitors interact with the website. These cookies help provide information on metrics such as the number of visitors, bounce rate, traffic source, etc.",
        "Performance cookies are used to understand and analyze the key performance indexes of the website which helps in delivering a better user experience for the visitors.",
        "No cookies to display.",
        "Advertisement cookies are used to provide visitors with customized advertisements based on the pages you visited previously and to analyze the effectiveness of the ad campaigns.",
        "Other uncategorized cookies are those that are being analyzed and have not been classified into a category as yet.",
        "Purposes & Features",
        "Cookies, device or similar online identifiers (e.g. login-based identifiers, randomly assigned identifiers, network based identifiers) together with other information (e.g. browser type and information, language, screen size, supported technologies etc.) can be stored or read on your device to recognise it each time it connects to an app or to a website, for one or several of the purposes presented here.",
        "Vendors Privacy Policy: Legitimate Interest Claim: Purposes (Consent) Features Privacy Policy: Legitimate Interest Claim: Purposes (Consent) Features Special Features Privacy Policy"
    ]
    normalized_exact_matches = set(s.lower().strip() for s in exact_matches)

    # Additional UI-related keywords (case-insensitive)
    ui_keywords = [
        "transparent semi-transparent opaque font size",
        "this is a modal window.",
        "this modal can be closed by pressing the escape key or activating the close button.",
        "escape will cancel and close the window",
        "play mute current time",
        "liveremaining time",
        "playback rate",
        "live seek",
        "video player is loading.",
        "play video",
        "play unmute",
        "selected captions",
        "opens captions settings",
        "dialogcaptions",
        "opaque font",
        "semi-transparent opaque font",
        "text font",
        "text color",
        "updates in your inbox!",
        "stay up-to-date on the latest",
        "read a summary of this article",
        "this audio is generated by an ai tool.",
        "transparency opaque semi-transparent background color"
    ]

    # Patterns for long consent/cookie disclosure sentences (case-insensitive)
    consent_patterns = [
        r"we value your privacy\s+we\s*and\s*our\s*\d{1,4}\s*partnersuse cookies",
        r"we may store and\/or access information on a device and process personal data,.*?personalised advertising and content",
        r"cookie categories\s+we use cookies to help you navigate efficiently and perform certain functions",
        r"necessary cookies are required to enable the basic features of this site",
        r"analytical cookies are used to understand how visitors interact with the website",
        r"cookies, device or similar online identifiers.*?for one or several of the purposes presented here",
        r"illustrations advertising presented to you on this service can be based on limited data",
        r"illustrations information about your activity on this service.*?build or improve a profile about you",
        r"your profile can be used.*?to present advertising that appears more relevant",
        r"illustrations content presented to you on this service can be based on your content personalisation profiles",
        r"information regarding which advertising is presented to you and how you interact with it can be used",
        r"information regarding which content is presented to you and how you interact with it can be used",
        r"illustrations reports can be generated based on the combination of data sets",
        r"information about your activity on this service, such as your interaction with ads or content, can be very helpful to improve products and services",
        r"your data can be used to monitor for and prevent unusual and possibly fraudulent activity",
        r"the choices you make regarding the purposes and entities listed in this notice are saved",
        r"information about your activity on this service may be matched and combined with other information relating to you",
        r"with your acceptance, your precise location.*?may be used in support of the purposes explained in this notice",
        r"with your acceptance, certain characteristics specific to your device might be requested and used"
    ]

    # Filter sentences based on exact matches, partial match keywords, consent patterns, and UI keywords.
    sentences = []
    for s in sentence_candidates:
        s_clean = s.strip()
        s_lower = s_clean.lower()

        if not s_clean:
            continue

        # Exact sentence match check
        if s_lower in normalized_exact_matches:
            continue

        # Check for any partial keyword match
        if any(re.search(pat, s_clean, flags=re.IGNORECASE) for pat in partial_match_keywords):
            continue

        # Check for any consent notice pattern match
        if any(re.search(pat, s_clean, flags=re.IGNORECASE) for pat in consent_patterns):
            continue

        # Check for any UI keyword (substring)
        if any(kw in s_lower for kw in ui_keywords):
            continue

        # Also skip sentences starting with "Click here" or "Post a comment"
        if re.match(r"(?i)^Click here", s_clean) or re.match(r"(?i)^Post a comment", s_clean):
            continue

        sentences.append(s_clean)

    # Rejoin sentences and fix minor casing issues (e.g., merged words like "wordWord")
    text = " ".join(sentences)
    text = re.sub(r"(?<=[a-z])(?=[A-Z][a-z])", " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()


def clean_full_text_column(df, text_column='full_text'):
    """
    Apply the custom_conservative_cleaner to a specified text column of a pandas DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing the text data.
        text_column (str): Name of the column with raw text to clean.

    Returns:
        pd.DataFrame: Modified DataFrame with a new 'clean_full_text' column.
    """
    df['clean_full_text'] = df[text_column].apply(custom_conservative_cleaner)
    return df


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [52]:
df_cleaned = clean_full_text_column(df)

In [55]:
df_cleaned[df_cleaned['clean_full_text'].apply(lambda x: len(x.split()) > 20000)]

,keyword,country,title,description,published_date,url,publisher,full_text,clean_full_text


In [58]:
df_cleaned['clean_full_text'].apply(lambda x: len(x.split())).max()

16942

In [59]:
# Output to parquet format
df_cleaned.to_parquet('cleaned_news_data.parquet', engine='pyarrow', index=False)